# GRPO: inspect examples or the full training split

Set `NUM_EXAMPLES` to `1`–`5` for a walkthrough, or `None` for the full GSM8K training split. `BATCH_SIZE` controls how many prompts are processed at once. Each prompt produces `GROUP_SIZE` sampled completions.

This notebook inspects sampling, rewards, advantages, token log probabilities, ratios, and policy loss. It computes gradients for the current policy but does **not** update model weights.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


In [3]:
# Support a kernel started in this notebook's folder or at the repository root.
module_dir = Path.cwd()
if not (module_dir / "data.py").is_file():
    module_dir = module_dir / "model_training" / "post_training" / "06-grpo" 
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

In [4]:
# if not (module_dir / "grpo" / "data.py").is_file():
#     raise FileNotFoundError("Start the kernel in 06-grpo or the deep_learning repository root.")

In [19]:
from grpo.data import make_prompt
from grpo.model import inspect_grpo_batch, generate_rollouts
from grpo.grpo_loss import (
    completion_token_logprobs,
    compute_advantages,
    compute_ratio,
    grpo_policy_loss,
)
from verifier import compute_group_rewards, classify_response


logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Choose the examples and generation settings

For the complete training split, set `NUM_EXAMPLES = None` and `START_INDEX = 0`. A small batch limits memory use; a full run still requires generating and scoring every completion.

In [6]:
MODEL_NAME = "Qwen/Qwen3-1.7B"
DATASET_NAME = "openai/gsm8k"
NUM_EXAMPLES = 10       # 1–5 for exploration; None for the full training split
START_INDEX = 4        # Dataset index of the first selected example
BATCH_SIZE = 1         # Prompts per batch, independent of GROUP_SIZE
GROUP_SIZE = 4         # Sampled completions per prompt
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.6
CLIP_EPS = 0.2

In [7]:

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
dtype = (
    torch.bfloat16 if device.type == "cuda"
    else torch.float16 if device.type == "mps"
    else torch.float32
)
{"device": str(device), "dtype": str(dtype), "batch_size": BATCH_SIZE, "group_size": GROUP_SIZE}

{'device': 'mps', 'dtype': 'torch.float16', 'batch_size': 1, 'group_size': 4}

In [8]:
if BATCH_SIZE < 1 or GROUP_SIZE < 1 or START_INDEX < 0:
    raise ValueError("BATCH_SIZE and GROUP_SIZE must be positive, and START_INDEX must be nonnegative")

In [9]:
if NUM_EXAMPLES is not None and NUM_EXAMPLES < 1:
    raise ValueError("NUM_EXAMPLES must be positive or None")

In [10]:
train_data = load_dataset(DATASET_NAME, "main", split="train")
if START_INDEX >= len(train_data):
    raise ValueError("START_INDEX is outside the training split")

In [11]:
end_index = len(train_data) if NUM_EXAMPLES is None else min(START_INDEX + NUM_EXAMPLES, len(train_data))
selected_data = train_data.select(range(START_INDEX, end_index))

In [12]:
pd.DataFrame([{
    "training_examples": len(train_data),
    "selected_examples": len(selected_data),
    "first_dataset_index": START_INDEX,
    "last_dataset_index": end_index - 1,
    "expected_completions": len(selected_data) * GROUP_SIZE,
}])

,training_examples,selected_examples,first_dataset_index,last_dataset_index,expected_completions
0,7473,10,4,13,40


## 2. Load the model and inspect the first prompt

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
first_example = selected_data[0]
display(pd.DataFrame([first_example]))
print(make_prompt(first_example["question"]))

,question,answer
0,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


Solve the following math problem.
Return only the final numerical answer.
Use the format: FINAL_ANSWER: <number>

Question: James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?


## 3. Inspect the first batch

The helper runs each GRPO stage for this batch. Generated IDs, attention masks, rewards, advantages, log probabilities, and ratios are available in `first_batch`.

In [15]:
first_batch_size = min(BATCH_SIZE, len(selected_data))
first_examples = [selected_data[i] for i in range(first_batch_size)]
first_examples

[{'question': 'James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?',
  'answer': 'He writes each friend 3*2=<<3*2=6>>6 pages a week\nSo he writes 6*2=<<6*2=12>>12 pages every week\nThat means he writes 12*52=<<12*52=624>>624 pages a year\n#### 624'}]

#### Step-1:- Generate Rollouts

In [16]:
input_ids, attention_mask, output_ids, predictions, sequence_mask = generate_rollouts(
        model=model,
        tokenizer=tokenizer,
        examples=first_examples,
        group_size=GROUP_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        device=device,
    )
print(predictions)

['James writes a 3-page letter to 2 different friends twice a week.\n\nFirst, calculate the number of letters he writes per week:\n$$\n2 \\text{ friends} \\times 2 \\text{ times per week} = 4 \\text{ letters per week}\n$$\n\nNext, calculate the number of pages per week:\n$$\n4 \\text{ letters} \\times 3 \\text{ pages per letter} = 12 \\text{ pages per week}\n$$\n\nNow calculate the number of pages per year (52 weeks):\n$$\n12 \\text{ pages per week} \\times', 'James writes a 3-page letter to 2 different friends twice a week.\n\nFirst, calculate the number of letters he writes per week:\n$$\n2 \\text{ friends} \\times 2 \\text{ times per week} = 4 \\text{ letters per week}\n$$\n\nNext, calculate the number of pages per week:\n$$\n4 \\text{ letters per week} \\times 3 \\text{ pages per letter} = 12 \\text{ pages per week}\n$$\n\nNow, calculate the number of pages per year (52 weeks):\n$$\n12 \\text{ pages per week}', 'James writes a 3-page letter to 2 different friends twice a week.\n\nF

#### Step-2: -  Compute Rewards and Advantage

In [17]:
# Step 3: Generate rewards and advantages.
rewards = compute_group_rewards(
    predictions, [example["answer"] for example in first_examples], GROUP_SIZE, device
)
print(rewards)

tensor([[0., 0., 0., 0.]], device='mps:0')


In [33]:
failure_reason = classify_response(predictions[0], first_examples[0]["answer"])
failure_reason

'invalid_format'

In [34]:
advantages = compute_advantages(rewards)
print(advantages)
prompt_len = input_ids.shape[1]
print(prompt_len)

tensor([[0., 0., 0., 0.]], device='mps:0')
65


#### Step-3:- Generate the probability of the computed tokens
    

In [35]:
with torch.no_grad():
    old_logits = model(output_ids, attention_mask=sequence_mask, use_cache=False).logits
    old_logps, mask = completion_token_logprobs(
        old_logits, output_ids, prompt_len, sequence_mask
    )
print(old_logits.shape)

torch.Size([4, 193, 151936])


In [36]:
del old_logits

#### Step-5: - Compute log probability of new tokens

In [37]:
with torch.enable_grad():
    current_logits = model(output_ids, attention_mask=sequence_mask, use_cache=False).logits
    logits_shape = tuple(current_logits.shape)
    current_logps, _ = completion_token_logprobs(
        current_logits, output_ids, prompt_len, sequence_mask
    )

In [38]:
ratio = compute_ratio(current_logps, old_logps)
ratio

tensor([[1.0000, 1.0000, 1.0078, 1.0010, 1.0000, 0.9922, 0.9844, 0.9990, 0.9922,
         1.0000, 0.9961, 1.0000, 1.0000, 1.0322, 1.0000, 0.9844, 1.0078, 1.0078,
         0.9844, 0.9951, 1.0000, 1.0156, 1.0000, 1.0000, 0.9863, 0.9829, 1.0000,
         1.0078, 1.0059, 1.0156, 1.0117, 0.9951, 1.0117, 1.0098, 1.0000, 1.0195,
         0.9976, 1.0273, 1.0000, 1.1152, 1.0156, 1.1416, 1.0000, 1.0010, 0.9990,
         1.0098, 0.9956, 1.0000, 0.9927, 0.9980, 1.0020, 1.0000, 1.0322, 0.9844,
         1.0000, 1.0322, 1.0137, 0.9883, 1.0645, 1.0000, 1.0000, 1.0156, 1.0000,
         1.0000, 1.0020, 1.0000, 0.9854, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0010, 1.0010,
         1.0000, 1.0000, 1.0000, 1.0020, 1.0000, 1.0000, 1.0234, 1.0000, 0.9971,
         1.0000, 1.0000, 1.0000, 0.9980, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         0.9995, 1.0000, 1.0

In [39]:
loss = grpo_policy_loss(current_logps, old_logps, advantages, mask, CLIP_EPS)
loss

tensor(0., device='mps:0', grad_fn=<DivBackward0>)

In [40]:
current_requires_grad = current_logps.requires_grad
current_requires_grad

True

In [41]:
first_batch = inspect_grpo_batch(
    model=model,
    tokenizer=tokenizer,
    examples=first_examples,
    group_size=GROUP_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    device=device,
    clip_eps=CLIP_EPS,
)

pd.DataFrame([{
    "input_ids_shape": tuple(first_batch["input_ids"].shape),
    "output_ids_shape": tuple(first_batch["output_ids"].shape),
    "logits_shape": first_batch["logits_shape"],
    "rewards_shape": tuple(first_batch["rewards"].shape),
    "advantages_shape": tuple(first_batch["advantages"].shape),
    "old_logps_require_grad": first_batch["old_requires_grad"],
    "current_logps_require_grad": first_batch["current_requires_grad"],
    "policy_loss": first_batch["loss"],
}])



,input_ids_shape,output_ids_shape,logits_shape,rewards_shape,advantages_shape,old_logps_require_grad,current_logps_require_grad,policy_loss
0,"(1, 65)","(4, 193)","(4, 193, 151936)","(1, 4)","(1, 4)",False,True,0.0


## 4. Completions, rewards, and advantages

Rows are ordered by prompt and then by completion. Rewards and advantages have shape `(number of prompts, group size)`; advantages are normalized within each prompt's group.

In [42]:
first_results = pd.DataFrame(first_batch["records"])
first_results["example_index"] += START_INDEX
first_results[["example_index", "completion_index", "question", "gold_answer", "prediction", "reward", "advantage", "batch_loss"]]

,example_index,completion_index,question,gold_answer,prediction,reward,advantage,batch_loss
0,4,0,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.0,0.0
1,4,1,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.0,0.0
2,4,2,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.0,0.0
3,4,3,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.0,0.0


In [43]:
display({"rewards": first_batch["rewards"], "advantages": first_batch["advantages"]})

{'rewards': tensor([[0., 0., 0., 0.]]),
 'advantages': tensor([[0., 0., 0., 0.]])}

## 5. Process the selected dataset in batches

The first batch is reused. Results for every completion are retained in `results_df`; token IDs, log probabilities, and ratios are stored per row so you can inspect any example. Logits and computation graphs are released after each batch.

In [44]:
results = []
batch_diagnostics = []

for start in tqdm(range(0, len(selected_data), BATCH_SIZE), desc="GRPO inspection batches"):
    batch_examples = [selected_data[i] for i in range(start, min(start + BATCH_SIZE, len(selected_data)))]
    batch = inspect_grpo_batch(
        model=model,
        tokenizer=tokenizer,
        examples=batch_examples,
        group_size=GROUP_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        device=device,
        clip_eps=CLIP_EPS,
    )
    for record in batch["records"]:
        results.append({**record, "example_index": START_INDEX + start + record["example_index"]})
    batch_diagnostics.append({
        "first_example_index": START_INDEX + start,
        "examples": len(batch_examples),
        "logits_shape": batch["logits_shape"],
        "policy_loss": batch["loss"],
        "old_logps_require_grad": batch["old_requires_grad"],
        "current_logps_require_grad": batch["current_requires_grad"],
    })

results_df = pd.DataFrame(results)
batch_diagnostics_df = pd.DataFrame(batch_diagnostics)
results_df[["example_index", "completion_index", "question", "gold_answer", "prediction", "reward", "advantage", "batch_loss"]]

GRPO inspection batches:   0%|          | 0/10 [00:00<?, ?it/s]

,example_index,completion_index,question,gold_answer,prediction,reward,advantage,batch_loss
0,4,0,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.000000,0.00000
1,4,1,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.000000,0.00000
2,4,2,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.000000,0.00000
3,4,3,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,James writes a 3-page letter to 2 different fr...,0.0,0.000000,0.00000
4,5,0,Mark has a garden with flowers. He planted pla...,There are 80/100 * 10 = <<80/100*10=8>>8 more ...,Let's break it down step by step:\n\n1. Mark h...,0.0,0.000000,0.00000
5,5,1,Mark has a garden with flowers. He planted pla...,There are 80/100 * 10 = <<80/100*10=8>>8 more ...,Let's break down the problem step by step:\n\n...,0.0,0.000000,0.00000
6,5,2,Mark has a garden with flowers. He planted pla...,There are 80/100 * 10 = <<80/100*10=8>>8 more ...,Let's break down the problem step by step.\n\n...,0.0,0.000000,0.00000
7,5,3,Mark has a garden with flowers. He planted pla...,There are 80/100 * 10 = <<80/100*10=8>>8 more ...,Let's break down the problem step by step.\n\n...,0.0,0.000000,0.00000
8,6,0,Albert is wondering how much pizza he can eat ...,He eats 32 from the largest pizzas because 2 x...,"Albert buys 2 large pizzas, each with 16 slice...",1.0,0.000000,0.00000
9,6,1,Albert is wondering how much pizza he can eat ...,He eats 32 from the largest pizzas because 2 x...,"Albert buys 2 large pizzas, each with 16 slice...",1.0,0.000000,0.00000


In [45]:
batch_diagnostics_df

,first_example_index,examples,logits_shape,policy_loss,old_logps_require_grad,current_logps_require_grad
0,4,1,"(4, 193, 151936)",0.00000,False,True
1,5,1,"(4, 232, 151936)",0.00000,False,True
2,6,1,"(4, 161, 151936)",0.00000,False,True
3,7,1,"(4, 273, 151936)",0.06805,False,True
4,8,1,"(4, 276, 151936)",0.00000,False,True
5,9,1,"(4, 234, 151936)",0.00000,False,True
6,10,1,"(4, 243, 151936)",0.00000,False,True
7,11,1,"(4, 264, 151936)",0.00000,False,True
8,12,1,"(4, 204, 151936)",0.00000,False,True
9,13,1,"(4, 220, 151936)",0.00000,False,True


## 6. Inspect tokens for any completion

Change the two indices to select a row from `results_df`. These arrays contain generated tokens through the first stop token, excluding prompt and padding positions.

In [ ]:
EXAMPLE_TO_INSPECT = START_INDEX
COMPLETION_TO_INSPECT = 0

matches = results_df.loc[
    (results_df["example_index"] == EXAMPLE_TO_INSPECT)
    & (results_df["completion_index"] == COMPLETION_TO_INSPECT)
]
if matches.empty:
    raise ValueError("The selected example and completion are not present in results_df")
row = matches.iloc[0]
display(row[["question", "gold_answer", "prediction", "reward", "advantage", "batch_loss"]])

pd.DataFrame({
    "token_id": row["token_ids"],
    "token": tokenizer.convert_ids_to_tokens(row["token_ids"].tolist()),
    "old_logprob": row["old_logps"],
    "current_logprob": row["current_logps"],
    "ratio": row["ratio"],
})

The old and current policy use the same weights here, so their ratios should be close to 1. The loss is an inspection value for each batch; no optimizer step is performed.

In [ ]:
results_df.iloc[5]['prediction']

In [ ]:
import re
from decimal import Decimal

def diagnose(row):
    gold = Decimal(
        row["gold_answer"].split("####")[-1].strip().replace(",", "")
    )

    match = re.search(
        r"FINAL_ANSWER:\s*(-?[\d,]+(?:\.\d+)?)",
        row["prediction"],
        flags=re.IGNORECASE,
    )

    parsed = (
        Decimal(match.group(1).replace(",", ""))
        if match else None
    )

    return pd.Series({
        "parsed_answer": parsed,
        "gold_number": gold,
        "numeric_correct": parsed == gold if parsed is not None else False,
    })

In [ ]:
diagnostics = results_df.join(results_df.apply(diagnose, axis=1))

print("Strict reward successes:", int(diagnostics["reward"].sum()))
print("Numeric successes:", int(diagnostics["numeric_correct"].sum()))

display(diagnostics[
    ["example_index", "prediction", "parsed_answer",
     "gold_number", "reward", "numeric_correct"]
])

In [ ]:
# print("Model:", model.config._name_or_path)
# print("Tokenizer:", tokenizer.name_or_path)

# example = train_data[4]

# messages = [
#     {"role": "user", "content": make_prompt(example["question"])}
# ]

# prompt = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=False,
# )

# print("Prompt ending:", repr(prompt[-300:]))

# inputs = tokenizer(
#     prompt,
#     return_tensors="pt",
#     add_special_tokens=False,
# ).to(device)

# model.eval()

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=256,
#         do_sample=False,
#         pad_token_id=tokenizer.pad_token_id,
#     )

# completion = outputs[0, inputs["input_ids"].shape[1]:]

# print(tokenizer.decode(
#     completion,
#     skip_special_tokens=False
# ))